# Solving mixed-integer problems as QUBO

Many optimization problems are defined with mized integers, such as rocket control problems where postion variables $x_t$ are continous variables no just binary. This doesn't cleaning map onto our QUBO annealers which require us to have $E=x^TQx$ where $x_i=\{0,1\}$. One of the main problems are encoding these continuous variables into binary, we'll follow this paper and its strategies.

<a id="ref-iftakher"></a>
[Iftakher et al. 2022] Iftakher, A., Türkay, M., & Hasan, M.M.F. *Solving mixed-integer problems as QUBO: Encodings, reformulations, and rolling-precision algorithm*. Computers & Chemical Engineering, 163 (2022). [DOI:10.1016/j.compchemeng.2022.107856](https://doi.org/10.1016/j.compchemeng.2022.107856)



We have some main encodings in 

1. Simplest mixed $\mathbb{Z}$ encoding

    Take $x_t\in\mathbb{Z}$ with bounds $x_t=\in \{L_t,L_t+1,\cdots,U_t\}\subset\mathbb{Z}$. We define the range $R_t=U_t-L_t$, then with $B$ bits we can express $2^B$ distinct values $\{0,1,\cdots,2^B-1\}$. Hence we need $2^B-1\geq R_t$ to cover the range, and inverting $B_t=\log_2(R_t+1)$

    Then we introduce binary variables $z_{t,0},z_{t,1},\cdots,z_{t,2^B_t-1}=\{0,1\}$ and we can encode our integer variable as binary with.

    $x_t=L_t+\sum_{j=0}^{B_t-1}2^b z_{t,b}$ 

2. Decimal place expansions

    First take a continuous varibale $x^L_t\leq x_T\leq x^U_t$ now we scale our variable such that $\tilde x_t\in[0,1]$. Now,
    
     $\tilde x_t =\frac{x_t-x^L_t}{x_t^U-x_t^L}$ where $x_t=x_m^L+(x_m^U-x_m^L)\tilde x_t$

    From here we need to approximate $\tilde x_t$ with a finite decimal expansion of $J$ digits.

    $\tilde x_t = \underbrace{\sum_{j=1}^J \sum_{i=1}^{I=9} 10^{-j}z_{i,j,t}}_\text{J decimal places} + 10^{-J} z_{t,m}$

    Now $\sum_{j=1}^J 9\cdot 10^{-j}=1-10^{-J}$ hence we need an endpoint $+0^{-J}z_{t,m}$ so we can represent $1.000$. This is the simplest formulation but requires $(9J+1)$ binary variables per continous variable, this scales with $\sim9$ per decimal place. There are more effecient encodings such as the *weighted encoding with four bits per digit*.



3. Weighted encoding with four bits per digit
    
    $\tilde x_t = \underbrace{\sum_{j=1}^J 10^{-j}d_{t,j}}_\text{J decimal places} + \underbrace{10^{-J}z_{t,m}}_\text{endpoint bit}\\
    ~~= \underbrace{\sum_{j=1}^J 10^{-j}\left(z_{t,j,1}+2z_{t,j,2}+3z_{t,j,3}+3z_{t,j,4}\right)}_\text{weighted encoding with four bits per digit} + \underbrace{10^{-J}z_{t,m}}_\text{endpoint bit}$
    
    This choice of 4 bit encoding can get from $0\rightarrow (1+2+3+3)=9$ with only $(4J+1)$


These truncations have errors of $\text{error}\leq 10^{-J}$



# Mixed-integer optimal control

## 1D rocket control

The goal here is to construct an energy function $E=\textbf{x}^TQ\mathbf{x}$, where x are binary.T p

The general structure will be: 

$H_A$ enforcing the dynamics of the problme $x_{t+1}=f(x_t,u_t)$,

$H_B$ minimising fuel, 

$b_t$ binary control variable,

$x_t$ state variables, these require our encoding as continous variables.

### The dynamics

Lets first quickly dicuss the newtonian mechanics required for the dynamics of the problem
Lets define our parameters: 

$x(t)=$ height,

$\dot x(t)=v(t)=$ velocity,

$u(t)=$ engine thrust.

Firstly lets pick the simplest modle of thrust the engine is either on or off,

$u(t)=T_\text{max}b(t)$.

Appliyng some newtonian mechanics to this 1D rocket problem,
$F=ma\\
    F=Tu_t-mg\\
    a_t=\frac{u}{m}u_t-g$

We're left with our two equations $\begin{cases}
    \dot x=v\\
    \dot v=\frac{T\text{max}}{m}b(t)-g
\end{cases}$

#### Time discretisation

We then need to discretise time with the euler method for some tine step $\Delta t$

$x_{k+1}=x_k+\Delta t \dot x_k\\
x_{k+1}=x_k+\Delta t v_k$

and

$v_{k+1}=v_k+\Delta t \dot v_k\\
v_{k+1}=v_k+\Delta t (\frac{T\text{max}}{m}b_k-g)$

So we have, $\begin{cases}
    x_{k+1}=x_k+\Delta t v_k \\
    v_{k+1}=v_k+\Delta t (\frac{T\text{max}}{m}b_k-g)
\end{cases}$

with $b_k\in\{0,1\}$ as our binary variables.

### Our problem

Now lets formulate this problem in a mixed-integer optimal control problem form, say our goal is: ***to land whilst firing the engine as few timesteps as possible***.

$ 
\min\sum_{k=0}^{N-1}b_k \\
~\text{s.t.}~x_{k+1}=x_k+\Delta t v_k \\
~~ ~~ ~~ ~~ v_{k+1}=v_k+\Delta t(\frac{T\text{max}}{m}b_k-g) \\
~~ ~~ ~~ ~~ x_N=0, ~~v_N=0\\
~~ ~~ ~~ ~~  b_k\in\{0,1\}
$



## Energy function

Now build ourselves a energy function for this problem,

$E=E_\text{fuel}+E_\text{end}+E_\text{dynamics}$

$E_\text{fuel}=\alpha\sum_{k=0}^{N-1}b_k$

$E_\text{end}=\sum_{k=0}^{N-1}(x_N)^2+(v_N)^2=0$

$E_\text{dynamics}=\beta(E_\text{x dynamics})+\gamma(E_\text{v dynamics})\\
 = \beta\left(\sum_{k=0}^{N-1}(x_{k+1}-x_k-\Delta t v_{k})\right)^2+\gamma\left(\sum_{k=0}^{N-1}(v_{k+1}-v_k-\Delta t(\frac{T\text{max}}{m}b_k-g))\right)^2$

So we have

$E= \alpha\sum_{k=0}^{N-1}b_k+ \beta\left(\sum_{k=0}^{N-1}(\underbrace{x_{k+1}}_\text{not binary}-\underbrace{x_k}_\text{not binary}-\Delta t \underbrace{v_{k}}_\text{not binary} )\right)^2+ \gamma\left(\sum_{k=0}^{N-1}(\underbrace{v_{k+1}}_\text{not binary}-\underbrace{v_k}_\text{not binary}-\Delta t(\frac{T\text{max}}{m}b_k-g))\right)^2$

### Weighted encoding with four bits per digit

$x_m=x_m^\text{min}+(x_m^\text{max}-x_m^\text{min})\tilde x_t\\
~~ =x_m^\text{min}+(x_m^\text{max}-x_m^\text{min})\left( \sum_{j=1}^J 10^{-j}\left(x_{t,j,1}+2x_{t,j,2}+3x_{t,j,3}+3x_{t,j,4}\right) + 10^{-J}x_{t,m}\right)$

and 

$v_m=v_m^\text{min}+(v_m^\text{max}-v_m^\text{min})\left( \sum_{j=1}^J 10^{-j}\left(v_{t,j,1}+2v_{t,j,2}+3v_{t,j,3}+3v_{t,j,4}\right) + 10^{-J}v_{t,m}\right)$

We can then build ouselves a 

$E=\textbf{z}^TQ\mathbf{z}+c$

with,

$\mathbf{z}=\left(b_0,\ldots,b_{N-1},\mathbf{x}_0,\ldots,\mathbf{x}_N,\mathbf{v}_0,\ldots,\mathbf{v}_N\right)^T$

$\mathbf{x}_k=\left(x_{k,m},x_{k,1,1},x_{k,1,2},x_{k,1,3},x_{k,1,4},\ldots,x_{k,J,1},x_{k,J,2},x_{k,J,3},v_{k,J,4}\right)$

and similarly,

$\mathbf{v}_k=\left(v_{k,1,1},v_{k,1,2},v_{k,1,3},v_{k,1,4},\ldots,v_{k,J,1},v_{k,J,2},{k,J,3},v_{k,J,4}\right).$


In [2]:
import sympy as sp

# ==========================
# Problem parameters
# ==========================

N = 2      # number of timesteps
J = 1       # decimal precision digits

alpha, beta, gamma = sp.symbols(
    "alpha beta gamma"
)

dt, Tmax, m, g = sp.symbols(
    "dt Tmax m g"
)

xmin, xmax = sp.symbols(
    "xmin xmax"
)

vmin, vmax = sp.symbols(
    "vmin vmax"
)


# ==========================
# Binary control variables
# ==========================

b = [
    sp.Symbol(f"b{k}")
    for k in range(N-1)
]


# ==========================
# Create weighted encoding bits
# ==========================

x_bits = {}
v_bits = {}

for k in range(N):

    x_bits[k] = {
        "endpoint": sp.Symbol(f"x{k}_m"),
        "digits": [
            [
                sp.Symbol(f"x{k}_{j}_{i}")
                for i in range(1,5)
            ]
            for j in range(1,J+1)
        ]
    }

    v_bits[k] = {
        "endpoint": sp.Symbol(f"v{k}_m"),
        "digits": [
            [
                sp.Symbol(f"v{k}_{j}_{i}")
                for i in range(1,5)
            ]
            for j in range(1,J+1)
        ]
    }



# ==========================
# Encoding function
# ==========================

def weighted_encode(bits, lower, upper):

    y = 0

    # decimal digits
    for j, digit in enumerate(bits["digits"], start=1):

        y += (
            10**(-j)
            *
            (
                digit[0]
                + 2*digit[1]
                + 3*digit[2]
                + 3*digit[3]
            )
        )

    # endpoint bit
    y += 10**(-J)*bits["endpoint"]

    # scale to physical range
    return lower + (upper-lower)*y



# ==========================
# Encode states
# ==========================

x = []
v = []

for k in range(N):

    x.append(
        weighted_encode(
            x_bits[k],
            xmin,
            xmax
        )
    )

    v.append(
        weighted_encode(
            v_bits[k],
            vmin,
            vmax
        )
    )



# ==========================
# Construct energy
# ==========================

# Fuel objective
Efuel = alpha*sum(b)


# Position dynamics
Edyn_x = 0

for k in range(N-1):

    rx = (
        x[k+1]
        -
        x[k]
        -
        dt*v[k]
    )

    Edyn_x += rx**2



# Velocity dynamics
Edyn_v = 0

for k in range(N-1):

    rv = (
        v[k+1]
        -
        v[k]
        -
        dt*(Tmax/m*b[k]-g)
    )

    Edyn_v += rv**2



# Total energy

E = (
    Efuel
    +
    beta*Edyn_x
    +
    gamma*Edyn_v
)


# Expand
E = sp.expand(E)



# ==========================
# Apply binary rule
# z^2 = z
# ==========================

binary_vars = list(E.free_symbols)

for z in binary_vars:
    E = E.subs(z**2, z)

E = sp.expand(E)



# ==========================
# Extract QUBO matrix
# ==========================

vars = sorted(
    list(E.free_symbols),
    key=str
)

n = len(vars)

Q = sp.zeros(n,n)


for i in range(n):

    # diagonal terms
    Q[i,i] = E.coeff(vars[i])

    # quadratic terms
    for j in range(i+1,n):

        coeff = E.coeff(
            vars[i]*vars[j]
        )

        Q[i,j] = coeff/2
        Q[j,i] = coeff/2



# ==========================
# Output
# ==========================

print("Number of binary variables:")
print(n)

print("\nBinary variables:")
print(vars)

print("\nQUBO matrix:")
sp.pprint(Q)

Number of binary variables:
32

Binary variables:
[Tmax, alpha, b0, beta, dt, g, gamma, m, v0_1_1, v0_1_2, v0_1_3, v0_1_4, v0_m, v1_1_1, v1_1_2, v1_1_3, v1_1_4, v1_m, vmax, vmin, x0_1_1, x0_1_2, x0_1_3, x0_1_4, x0_m, x1_1_1, x1_1_2, x1_1_3, x1_1_4, x1_m, xmax, xmin]

QUBO matrix:
⎡  2⋅b₀⋅dt⋅g⋅γ   0.2⋅b₀⋅dt⋅γ⋅v₀ ₁ ₁⋅vmax   0.2⋅b₀⋅dt⋅γ⋅v₀ ₁ ₁⋅vmin   0.4⋅b₀⋅dt ↪
⎢- ─────────── + ─────────────────────── - ─────────────────────── + ───────── ↪
⎢       m                   m                         m                        ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                    dt⋅g⋅γ   0.1⋅dt⋅γ⋅v₀ ₁ ₁⋅vmax   0.1⋅dt⋅γ⋅ ↪
⎢                                  - ────── + ──────────────────── - ───────── ↪
⎢                                      m               m               